# Project: Discovering wine cultivars

*ML Academy — starter notebook*

**Goal.** Pretend we don't know the cultivars. Can PCA and clustering rediscover the three groups from the chemistry alone?

**The data.** 178 wines from three grape cultivars grown in the same region of Italy (Forina et al.), shipped with scikit-learn as load_wine. Thirteen chemical measurements per wine: alcohol, malic acid, flavanoids, colour intensity, proline and more. The cultivar labels exist — but we hide them until the very end.

**Uses:** Lesson 16, Lesson 17.

Work through the steps in order. Lines marked `# TODO` are yours to write; each step ends with a check that prints ✅ when you're right (and a hint when you're not). Cells that use a step's result will show errors until that step is done — that's expected. The solution notebook is in the same folder.

Requirements: `pip install numpy pandas scikit-learn matplotlib scipy` — the dataset ships with scikit-learn, so no download is needed.

In [1]:
# Run this cell first: it defines check(), which tells you whether each step is right.
import numpy as np
def check(get, want, hint=''):
    try:
        got = get()
        if isinstance(want, str):
            ok = str(got) == want
        else:
            ok = np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=1e-6)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


## Step 1. Load the chemistry (and put the labels aside)

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
wine = load_wine(as_frame=True)
X = wine.data                # 13 chemical measurements
hidden_labels = wine.target  # we won't look at these until the end
print(X.shape)
X.describe().loc[["mean", "std"]].round(2)

(178, 13)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
mean,13.00,2.34,2.37,19.49,99.74,2.30,2.03,0.36,1.59,5.06,0.96,2.61,746.89
std,0.81,1.12,0.27,3.34,14.28,0.63,1.00,0.12,0.57,2.32,0.23,0.71,314.91


In [3]:
check(lambda: X.shape, [178, 13], hint='load_wine(as_frame=True).data.')

✅ Correct!


## Step 2. Standardise

The features have wildly different scales (proline is in the hundreds, hue around 1). Standardise them so each counts equally (Lesson 17).

In [4]:
from sklearn.preprocessing import StandardScaler
Z = None  # TODO: standardise X
print(Z.mean(axis=0).round(2)[:4], Z.std(axis=0).round(2)[:4])

AttributeError: 'NoneType' object has no attribute 'mean'

In [5]:
check(lambda: float(np.round(Z.std(axis=0).mean(), 6)), 1.0, hint='StandardScaler().fit_transform(X) gives every column mean 0 and standard deviation 1.')

❌ Not yet. Hint: StandardScaler().fit_transform(X) gives every column mean 0 and standard deviation 1.


## Step 3. PCA: how many directions matter?

Fit PCA and look at the explained variance ratio of the first components.

In [6]:
from sklearn.decomposition import PCA
pca = PCA().fit(Z)
ratio = None  # TODO: the explained variance ratio of every component
print("first five:", ratio[:5].round(3))
print("cumulative:", np.cumsum(ratio)[:5].round(3))

ValueError: Expected 2D array, got scalar array instead:
array=nan.
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [7]:
check(lambda: ratio[0], 0.36198848099926334, hint='pca.explained_variance_ratio_.')
check(lambda: np.cumsum(ratio)[1], 0.5540633835693527, hint='The first two components together.')

❌ Not yet. Hint: pca.explained_variance_ratio_.
❌ Not yet. Hint: The first two components together.


## Step 4. Look at the data in 2D

Project onto the first two components and plot. Can you already see groups?

In [8]:
import matplotlib.pyplot as plt
P = PCA(n_components=2).fit_transform(Z)
plt.figure(figsize=(5, 4))
plt.scatter(P[:, 0], P[:, 1], s=15, color="grey")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("178 wines, unlabelled")
plt.show()

ValueError: Expected 2D array, got scalar array instead:
array=nan.
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

## Step 5. Choose k

Run k-means for k = 2 to 6 and compare inertia (elbow) and the silhouette score (Lesson 16).

In [9]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
sil = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    sil[k] = None  # TODO: the silhouette score of this clustering
    print(f"k={k}  inertia={km.inertia_:7.1f}  silhouette={sil[k]:.3f}")
best_k = max(sil, key=sil.get)
print("best k by silhouette:", best_k)

ValueError: Expected 2D array, got scalar array instead:
array=nan.
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [10]:
check(lambda: best_k, 3, hint='silhouette_score(Z, km.labels_) for each k.')
check(lambda: sil[3], 0.2848589191898987, hint='Use n_init=10 and random_state=0.')

❌ Not yet. Hint: silhouette_score(Z, km.labels_) for each k.
❌ Not yet. Hint: Use n_init=10 and random_state=0.


## Step 6. Reveal the labels

Now compare the clusters with the real cultivars. The adjusted Rand index (ARI) is 1 for a perfect match and about 0 for random clusters.

In [11]:
from sklearn.metrics import adjusted_rand_score
clusters = KMeans(n_clusters=best_k, n_init=10, random_state=0).fit_predict(Z)
ari = None  # TODO: the adjusted Rand index between hidden_labels and clusters
print("ARI:", round(ari, 3))
pd.crosstab(clusters, hidden_labels, rownames=["cluster"], colnames=["cultivar"])

NameError: name 'best_k' is not defined

In [12]:
check(lambda: ari, 0.8974949815093207, hint='adjusted_rand_score(hidden_labels, clusters).')

❌ Not yet. Hint: adjusted_rand_score(hidden_labels, clusters).


## Go further

- Repeat the clustering on the first two PCA components only. Is the ARI better or worse?
- Try agglomerative clustering and draw a dendrogram with SciPy.
- What happens to the silhouette scores if you forget to standardise?